# 03 · Modelos mixtos y deep learning longitudinal

| | |
|---|---|
| **Objetivo** | Identificar **predictores longitudinales** de gravedad y comparar un modelo de deep learning (LSTM) contra baselines simples |
| **Preguntas guía** | ¿Qué medidas de voz se asocian con la gravedad *entre* pacientes y cuáles *dentro* de cada paciente a lo largo del tiempo? ¿Una LSTM pronostica mejor que reglas simples? |
| **Punto de partida** | Notebook 02: en pacientes nuevos ningún modelo supera al promedio; conocer la línea de base del paciente es lo que más ayuda |
| **Datos** | Promedios semanales por paciente (966 semanas-paciente), para reducir el ruido de la voz |

**Índice**
0. Configuración
1. Resumen ejecutivo
2. Datos semanales y descomposición entre / dentro
3. Modelo mixto: estructura de la variación
4. Modelo mixto: predictores entre y dentro de pacientes
5. Diagnóstico del modelo y chequeos de robustez
6. Pronóstico a 4 semanas: baselines vs. Ridge vs. LSTM
7. Conclusiones e implicancias

> Requiere `statsmodels` y `torch`. Instalación (CPU): `pip install statsmodels torch --index-url https://download.pytorch.org/whl/cpu`

## 0. Configuración

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from src.analysis import ID
from src.data import load_data
from src.modeling import agregar_semanal
from src.longitudinal import (VOZ_SELECCION, preparar_datos_mixto, formula_mixto, ajustar_mixto,
                              tabla_coeficientes, componentes_varianza, tabla_diagnostico,
                              chequeos_robustez)
from src.deep import crear_ventanas, evaluar_pronostico, LARGO, HORIZONTE
from src.resumen03 import metricas_clave_03, hallazgos_clave_03
from src.viz import guardar, tarjetas_kpi, panel_diagnostico

pd.set_option("display.float_format", "{:.3f}".format)
OBJETIVO = "total_UPDRS"

In [2]:
df = load_data()
semanal = agregar_semanal(df)
d = preparar_datos_mixto(semanal, OBJETIVO)

print("Ajustando modelo mixto...")
resultado = ajustar_mixto(d)
coefs = tabla_coeficientes(resultado)
varianzas = componentes_varianza(resultado)
diagnostico = tabla_diagnostico(d, resultado)
robustez = chequeos_robustez(d)

print("Pronóstico con ventanas temporales (incluye entrenamiento de la LSTM)...")
ventanas = crear_ventanas(semanal, OBJETIVO)
pronostico, predicciones = evaluar_pronostico(ventanas)
print("Listo.")

Ajustando modelo mixto...
Pronóstico con ventanas temporales (incluye entrenamiento de la LSTM)...
  fold 1/5


  fold 2/5


  fold 3/5


  fold 4/5


  fold 5/5


Listo.


## 1. Resumen ejecutivo

In [3]:
m = metricas_clave_03(coefs, varianzas, pronostico, diagnostico, robustez)

kpis = tarjetas_kpi([
    ("Varianza entre pacientes al inicio", m["icc_inicio"], ".0%"),
    ("Progresión promedio (UPDRS/mes)", m["pendiente_media"], ".2f"),
    ("Predictores de voz dentro del paciente (p<0,05)", len(m["sig_dentro"]), ".0f"),
    ("MAE · mejor pronóstico a 4 semanas", m["mejor_mae"], ".2f"),
])
guardar(kpis, "03_kpis")
kpis.show()

display(Markdown("### Hallazgos clave\n\n" +
                 "\n".join(f"{i}. {h}" for i, h in enumerate(hallazgos_clave_03(m), 1))))

### Hallazgos clave

1. **Cada paciente tiene su propio nivel y su propio ritmo.** Al inicio, el 98% de la varianza no explicada por los predictores es entre pacientes. La pendiente promedio es de +0.57 puntos de UPDRS por mes (IC 95%: +0.17 a +0.96; p = 0.005), pero varía mucho entre pacientes: desvío de 1.30 puntos/mes.
2. **Ninguna medida de voz distingue de forma significativa a pacientes más y menos graves** una vez ajustado por edad, sexo y las demás medidas. Es coherente con el notebook 02: para un paciente nuevo, ningún modelo supera a predecir el promedio.
3. **Ninguna medida de voz funciona como predictor longitudinal:** cuando la voz de un paciente cambia respecto de su propio promedio, su UPDRS no cambia de forma significativa.
4. **El deep learning no supera a una regla simple:** pronosticando 4 semanas hacia adelante en pacientes nuevos, el mejor modelo es *Extrapolación lineal* (MAE 0.31). La LSTM obtiene 1.68, y agregar voz a Ridge pasa de 0.62 a 0.70. Con 42 pacientes y una UPDRS interpolada, la estructura de los datos favorece a la extrapolación lineal.
5. **Los supuestos se revisaron, y las conclusiones resisten.** El modelo converge. Los residuos tienen una autocorrelación semana a semana de +0.93 (la UPDRS está interpolada), por lo que los p-valores *dentro del paciente* pueden ser optimistas: como igual ninguno es significativo, la conclusión no cambia. Con métodos que no dependen de esos supuestos, la progresión sigue siendo significativa (p = 0.007) y ninguna medida de voz resulta significativa entre pacientes.

## 2. Datos semanales y descomposición entre / dentro

**Por qué semanal:** la voz varía mucho de una grabación a otra (notebook 01). Promediar por semana reduce ese ruido.

**Selección de variables:** una medida por familia (`Jitter(Abs)`, `Shimmer:APQ11`, `HNR`, `RPDE`, `DFA`, `PPE`),
para evitar la redundancia del notebook 01.

**Descomposición entre / dentro:** cada medida (estandarizada) se separa en dos columnas:

| Componente | Cálculo | Pregunta que responde |
|---|---|---|
| `_entre` | promedio del paciente | ¿Los pacientes con este rasgo son, en general, más graves? |
| `_dentro` | valor de la semana − promedio del paciente | Cuando ESTE paciente cambia, ¿cambia su gravedad? → **predictor longitudinal** |

In [4]:
cols_hnr = ["subject_id", "semanas", "y", "hnr", "hnr_entre", "hnr_dentro"]
d[cols_hnr].head(8)

,subject_id,semanas,y,hnr,hnr_entre,hnr_dentro
0,1,0,34.398,27.765,1.018,0.622
1,1,1,34.894,25.900,1.018,0.119
2,1,2,35.389,25.843,1.018,0.104
3,1,3,35.810,28.148,1.018,0.726
4,1,4,36.375,25.558,1.018,0.027
5,1,5,36.870,25.657,1.018,0.054
6,1,6,37.363,23.689,1.018,-0.476
7,1,7,37.857,24.385,1.018,-0.289


In [5]:
print(f"{len(d)} semanas-paciente · {d[ID].nunique()} pacientes · "
      f"{d.groupby(ID).size().median():.0f} semanas por paciente (mediana)")
print("\nFórmula del modelo:\n", formula_mixto())

966 semanas-paciente · 42 pacientes · 23 semanas por paciente (mediana)

Fórmula del modelo:
 y ~ meses + edad_z + sex + jitter_abs_entre + jitter_abs_dentro + shimmer_apq11_entre + shimmer_apq11_dentro + hnr_entre + hnr_dentro + rpde_entre + rpde_dentro + dfa_entre + dfa_dentro + ppe_entre + ppe_dentro


## 3. Modelo mixto: estructura de la variación

**Qué es un modelo mixto:** una regresión que reconoce que las semanas de un mismo paciente no son independientes. Además de los
**efectos fijos** (iguales para todos), estima **efectos aleatorios** por paciente:

- **Intercepto aleatorio:** cada paciente arranca en su propio nivel de gravedad.
- **Pendiente aleatoria:** cada paciente progresa a su propio ritmo.

Es la herramienta estándar para datos clínicos longitudinales, y los resultados son interpretables (coeficientes con intervalos de confianza).

**El tiempo entra en meses:** con el tiempo en semanas, la varianza de la pendiente es tan chica comparada con la del intercepto
que el optimizador **no converge**. Un modelo que no convergió da errores estándar poco confiables (en este caso subestimaba
la varianza entre pacientes y hacía parecer significativas medidas de voz que no lo son). En meses converge sin avisos.

In [6]:
print("Convergió:", resultado.converged)
print(resultado.summary())

Convergió: True
              Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    y         
No. Observations:    966        Method:                REML      
No. Groups:          42         Scale:                 2.1939    
Min. group size:     17         Log-Likelihood:        -1961.8132
Max. group size:     28         Converged:             Yes       
Mean group size:     23.0                                        
-----------------------------------------------------------------
                      Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
-----------------------------------------------------------------
Intercept             27.989    2.009 13.929 0.000  24.051 31.927
meses                  0.566    0.203  2.786 0.005   0.168  0.964
edad_z                 2.071    1.620  1.278 0.201  -1.105  5.247
sex                   -2.732    3.715 -0.735 0.462 -10.013  4.549
jitter_abs_entre      -5.758    4.619 -1.247 0.213 -14.812  3.295
jitter_a

In [7]:
pd.Series(varianzas, name="valor").to_frame()

,valor
var_intercepto,106.065
sd_pendiente,1.303
var_residuo,2.194
icc_inicio,0.980


In [8]:
# Pendiente de cada paciente = pendiente promedio (efecto fijo) + su desvío (efecto aleatorio)
pendientes = pd.DataFrame({
    ID: list(resultado.random_effects),
    "pendiente": [resultado.fe_params["meses"] + re.iloc[1] for re in resultado.random_effects.values()],
})
fig = px.histogram(pendientes, x="pendiente", nbins=15,
                   title="Ritmo de progresión estimado por paciente",
                   labels={"pendiente": "Cambio de UPDRS por mes"})
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.add_vline(x=resultado.fe_params["meses"], line_color="black",
              annotation_text="Promedio")
guardar(fig, "03_pendientes_por_paciente")
fig

In [9]:
d["ajustado"] = resultado.fittedvalues
ejemplos = pendientes.sort_values("pendiente")[ID].iloc[[0, 1, 20, 21, -2, -1]].tolist()
muestra = d[d[ID].isin(ejemplos)].melt(id_vars=[ID, "semanas"], value_vars=["y", "ajustado"],
                                        var_name="serie", value_name="UPDRS")
muestra["serie"] = muestra["serie"].map({"y": "Observado", "ajustado": "Modelo mixto"})
fig = px.line(muestra, x="semanas", y="UPDRS", color="serie", facet_col=ID, facet_col_wrap=3,
              markers=True, title="Observado vs. ajustado: pacientes con progresión lenta, media y rápida")
fig.update_layout(height=550)
guardar(fig, "03_ajuste_ejemplos")
fig

## 4. Predictores entre y dentro de pacientes

Coeficientes en **puntos de UPDRS por cada desvío estándar** de la medida de voz, con intervalo de confianza del 95%.
Si el intervalo cruza el 0, no hay evidencia de asociación.

In [10]:
coefs[["variable", "nivel", "coef", "ic_inf", "ic_sup", "p_valor", "significativo"]]

,variable,nivel,coef,ic_inf,ic_sup,p_valor,significativo
0,Intercept,Otros,27.989,24.051,31.927,0.000,True
1,meses,Otros,0.566,0.168,0.964,0.005,True
2,edad_z,Otros,2.071,-1.105,5.247,0.201,False
3,sex,Otros,-2.732,-10.013,4.549,0.462,False
4,Jitter(Abs),Entre pacientes,-5.758,-14.812,3.295,0.213,False
5,Jitter(Abs),Dentro del paciente,0.017,-0.311,0.345,0.918,False
6,Shimmer:APQ11,Entre pacientes,-0.727,-9.500,8.045,0.871,False
7,Shimmer:APQ11,Dentro del paciente,-0.152,-0.430,0.126,0.284,False
8,HNR,Entre pacientes,-0.784,-11.118,9.550,0.882,False
9,HNR,Dentro del paciente,0.089,-0.334,0.512,0.681,False


In [11]:
voz = coefs[coefs["nivel"] != "Otros"].copy()
voz["error_sup"] = voz["ic_sup"] - voz["coef"]
voz["error_inf"] = voz["coef"] - voz["ic_inf"]
voz["evidencia"] = voz["significativo"].map({True: "p < 0,05", False: "No significativo"})
fig = px.scatter(voz, x="coef", y="variable", color="nivel", error_x="error_sup", error_x_minus="error_inf",
                 symbol="evidencia", symbol_map={"p < 0,05": "circle", "No significativo": "circle-open"},
                 title=f"Asociación de la voz con {OBJETIVO}: entre pacientes vs. dentro del paciente",
                 labels={"coef": "Coeficiente (puntos de UPDRS por 1 desvío estándar)", "variable": ""})
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_traces(marker_size=10)
fig.update_layout(height=500)
guardar(fig, "03_coeficientes_entre_dentro")
fig

**Cómo leerlo:** los puntos **llenos** son significativos (p < 0,05). Un predictor longitudinal útil necesita
un efecto **dentro del paciente** claramente distinto de 0. Los efectos **entre pacientes** suelen tener intervalos más anchos,
porque con 42 pacientes hay solo 42 promedios para estimarlos.

## 5. Diagnóstico del modelo y chequeos de robustez

**Por qué:** los p-valores e intervalos del modelo mixto solo son confiables si se cumplen sus supuestos.
Antes de interpretar los coeficientes, se revisa:

| Supuesto | Cómo se revisa |
|---|---|
| El optimizador convergió | `resultado.converged` |
| Efectos aleatorios aproximadamente normales | Shapiro-Wilk y QQ-plot (42 valores, uno por paciente) |
| Residuos aproximadamente normales y con varianza constante | QQ-plot y residuos vs. ajustados |
| Residuos independientes dentro del paciente | Correlación del residuo de cada semana con el de la anterior |

Con ~1.000 residuos no se usa Shapiro-Wilk: con tantos datos rechaza la normalidad ante desvíos mínimos que no afectan
las conclusiones. El QQ-plot muestra *cuánto* se aparta y *dónde*.

In [12]:
diagnostico

,valor,cómo leerlo
chequeo,,
Convergencia del optimizador,Sí,Necesaria para confiar en errores estándar y p...
Efectos aleatorios · intercepto (Shapiro p),0.30,"p > 0,05: compatible con normalidad"
Efectos aleatorios · pendiente (Shapiro p),0.13,"p > 0,05: compatible con normalidad"
Residuos · asimetría,-0.02,Cerca de 0: simétricos
Residuos · exceso de curtosis,4.7,> 0: colas más pesadas que la normal
Residuos · |residuo| vs. ajustado (Spearman),+0.03,Cerca de 0: varianza constante
Residuos · autocorrelación semana a semana,+0.93,Cerca de 0: independientes dentro del paciente


In [13]:
r = d.assign(residuo=resultado.resid, ajustado=resultado.fittedvalues).sort_values([ID, "semanas"])
interceptos = pd.DataFrame(resultado.random_effects).T.iloc[:, 0]
fig = panel_diagnostico(r["residuo"], r["ajustado"], interceptos, r.groupby(ID)["residuo"].shift())
guardar(fig, "03_diagnostico_residuos", alto=800)
fig

**Cómo leerlo:**
- **Efectos aleatorios:** los interceptos siguen la recta del QQ-plot; el supuesto se cumple.
- **Residuos:** simétricos y con varianza constante, pero con **colas más pesadas** que una normal (los puntos se separan de la
  recta en los extremos). Con ~1.000 observaciones, esto afecta poco a los coeficientes.
- **Independencia: este es el supuesto que no se cumple.** El residuo de una semana es casi igual al de la siguiente
  (gráfico de abajo a la derecha). Es consecuencia de la UPDRS interpolada: entre evaluaciones clínicas, la gravedad sigue una recta.
  Con residuos autocorrelacionados, los p-valores *dentro del paciente* pueden ser optimistas.

**Chequeos de robustez:** se repiten las conclusiones con métodos que no dependen de estos supuestos.
1. **Progresión:** una recta por paciente y un test t sobre las 42 pendientes. Cada paciente aporta un solo número, así que la
   autocorrelación dentro del paciente no influye.
2. **Efectos entre pacientes:** una regresión con una fila por paciente (sus promedios).

In [14]:
robustez

,método,estimado,p_valor
conclusión,,,
Progresión promedio (UPDRS por mes),Recta por paciente + t de Student,0.573,0.007
Efecto entre pacientes · Jitter(Abs),Regresión con 1 fila por paciente,-5.521,0.238
Efecto entre pacientes · Shimmer:APQ11,Regresión con 1 fila por paciente,-0.885,0.845
Efecto entre pacientes · HNR,Regresión con 1 fila por paciente,-1.099,0.836
Efecto entre pacientes · RPDE,Regresión con 1 fila por paciente,2.681,0.448
Efecto entre pacientes · DFA,Regresión con 1 fila por paciente,-2.718,0.185
Efecto entre pacientes · PPE,Regresión con 1 fila por paciente,6.434,0.179


**Resultado:** las conclusiones se mantienen. La progresión promedio es significativa con los dos métodos, y ninguna medida de voz
resulta significativa, ni entre pacientes ni dentro del paciente. La autocorrelación de los residuos tiende a hacer *más* fácil encontrar
efectos significativos, no menos, así que el resultado nulo dentro del paciente es conservador.

## 6. Pronóstico a 4 semanas: baselines vs. Ridge vs. LSTM

**Tarea:** con las últimas **4 semanas** de un paciente (UPDRS + voz), predecir su UPDRS **4 semanas después**.
Se evalúa con `GroupKFold`: la predicción siempre es sobre pacientes que el modelo no vio.

Algunos pacientes tienen semanas sin grabaciones. Solo se usan ventanas con **4 semanas consecutivas** de historia y un objetivo
a **exactamente 4 semanas**; si no, un hueco convertiría el horizonte en 5 a 9 semanas.

| Modelo | Qué hace |
|---|---|
| Persistencia | "Nada cambia": repite el último valor |
| Extrapolación lineal | Prolonga la tendencia de las 4 semanas |
| Ridge · historial UPDRS | Regresión lineal sobre las 4 semanas de UPDRS |
| Ridge · historial UPDRS + voz | Ídem, sumando las 6 medidas de voz |
| **LSTM** · historial UPDRS + voz | Red recurrente que lee la secuencia semana a semana (PyTorch) |

**Cómo funciona la LSTM:** procesa una semana por vez y mantiene un "estado" (una memoria interna de 16 valores)
que resume lo visto hasta ese momento. Al terminar la secuencia, una capa lineal convierte ese estado en la predicción del cambio de UPDRS.

In [15]:
print(f"Ventanas: {ventanas['X'].shape[0]} · forma de cada entrada: {ventanas['X'].shape[1:]} "
      f"(semanas × variables: {ventanas['columnas']})")
pronostico

Ventanas: 465 · forma de cada entrada: (4, 7) (semanas × variables: ['total_UPDRS', 'Jitter(Abs)', 'Shimmer:APQ11', 'HNR', 'RPDE', 'DFA', 'PPE'])


,modelo,MAE,MAE_cambio_real_grande
0,Persistencia (no cambia),1.510,2.657
1,Extrapolación lineal,0.309,0.273
2,Ridge · historial UPDRS,0.620,0.979
3,Ridge · historial UPDRS + voz,0.700,1.046
4,LSTM · historial UPDRS + voz,1.680,2.412


In [16]:
largo = (pronostico.rename(columns={"MAE": "Todas las ventanas",
                                    "MAE_cambio_real_grande": "Solo cambios reales ≥ 2 puntos"})
         .melt(id_vars="modelo", var_name="evaluacion", value_name="MAE"))
fig = px.bar(largo, y="modelo", x="MAE", color="evaluacion", barmode="group", orientation="h", text_auto=".2f",
             title="Error del pronóstico a 4 semanas en pacientes nuevos (MAE, menor es mejor)",
             labels={"MAE": "MAE (puntos de UPDRS)", "modelo": "", "evaluacion": ""})
fig.update_layout(yaxis={"autorange": "reversed"}, height=450)
guardar(fig, "03_pronostico_mae")
fig

In [17]:
pid = predicciones.groupby("paciente")["real"].agg(lambda s: s.max() - s.min()).idxmax()
p = predicciones[predicciones["paciente"] == pid].reset_index(drop=True)
p["ventana"] = range(len(p))
serie = p.melt(id_vars="ventana", value_vars=["real", "Extrapolación lineal", "LSTM · historial UPDRS + voz",
                                             "Persistencia (no cambia)"], var_name="serie", value_name="UPDRS")
fig = px.line(serie, x="ventana", y="UPDRS", color="serie", markers=True,
              title=f"Paciente {pid} (el de mayor variación): pronóstico a 4 semanas vs. real")
guardar(fig, "03_pronostico_ejemplo")
fig

## 7. Conclusiones e implicancias

In [18]:
h = hallazgos_clave_03(m)
display(Markdown("**Lo que encontré**\n\n" + "\n".join(f"{i}. {x}" for i, x in enumerate(h, 1))))

**Lo que encontré**

1. **Cada paciente tiene su propio nivel y su propio ritmo.** Al inicio, el 98% de la varianza no explicada por los predictores es entre pacientes. La pendiente promedio es de +0.57 puntos de UPDRS por mes (IC 95%: +0.17 a +0.96; p = 0.005), pero varía mucho entre pacientes: desvío de 1.30 puntos/mes.
2. **Ninguna medida de voz distingue de forma significativa a pacientes más y menos graves** una vez ajustado por edad, sexo y las demás medidas. Es coherente con el notebook 02: para un paciente nuevo, ningún modelo supera a predecir el promedio.
3. **Ninguna medida de voz funciona como predictor longitudinal:** cuando la voz de un paciente cambia respecto de su propio promedio, su UPDRS no cambia de forma significativa.
4. **El deep learning no supera a una regla simple:** pronosticando 4 semanas hacia adelante en pacientes nuevos, el mejor modelo es *Extrapolación lineal* (MAE 0.31). La LSTM obtiene 1.68, y agregar voz a Ridge pasa de 0.62 a 0.70. Con 42 pacientes y una UPDRS interpolada, la estructura de los datos favorece a la extrapolación lineal.
5. **Los supuestos se revisaron, y las conclusiones resisten.** El modelo converge. Los residuos tienen una autocorrelación semana a semana de +0.93 (la UPDRS está interpolada), por lo que los p-valores *dentro del paciente* pueden ser optimistas: como igual ninguno es significativo, la conclusión no cambia. Con métodos que no dependen de esos supuestos, la progresión sigue siendo significativa (p = 0.007) y ninguna medida de voz resulta significativa entre pacientes.

**Limitaciones**
- **UPDRS interpolada** entre evaluaciones clínicas (inicio, 3 y 6 meses): genera trayectorias casi lineales, que favorecen a la extrapolación y dejan poca variación real dentro del paciente para asociar con la voz.
- **42 pacientes:** pocos para estimar efectos entre pacientes y para entrenar redes neuronales. Una LSTM necesita, en general, muchas más secuencias independientes.
- **Residuos autocorrelacionados:** por la interpolación, el residuo de una semana se parece mucho al de la siguiente. El modelo mixto
  no lo modela; por eso las conclusiones principales se repitieron con métodos que no dependen de ese supuesto (sección 5).
- **Selección de variables:** usamos una medida por familia; otras combinaciones podrían cambiar los efectos individuales.
- Asociación no implica causalidad: los coeficientes describen relaciones, no mecanismos.

**Qué me llevo para un proyecto real**
- **Empezar por un modelo mixto con descomposición entre / dentro:** responde directamente "¿qué predice el cambio en cada paciente?", es interpretable y es el estándar en investigación clínica.
- **El deep learning se justifica** cuando hay muchos pacientes, mediciones frecuentes y desenlaces medidos de verdad (no interpolados), o datos crudos como señales de sensores o imágenes. Y siempre se compara contra baselines simples.
- **Definir bien el desenlace:** en termorregulación, por ejemplo, los ítems de SCOPA-AUT por visita. Y considerar un **análisis de supervivencia** si la pregunta es *cuándo* aparece el trastorno.